In [ ]:
!pip install datasets


In [ ]:
from datasets import load_dataset

In [ ]:
persian_dataset = load_dataset("SeyedAli/Persian-Text-Sentiment")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/55852 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/13964 [00:00<?, ? examples/s]

In [ ]:
persian_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 55852
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 13964
    })
})

In [ ]:
min(persian_dataset['train']['label'])

0

In [ ]:
max(persian_dataset['train']['label'])

1

In [ ]:
for doc_index in range(0,10):
  print(f"[{doc_index}]\t label:{persian_dataset['train'][doc_index]['label']} \t text:{persian_dataset['train'][doc_index]['text'][0:100]}")

[0]	 label:1 	 text:این هندس فری تازه به دستم رسیده  و برای کسایی که انتظار شنیدن یک موسیقی با کیفیت دارن حتما پیشنهاد م
[1]	 label:1 	 text:عالی بود عالی، امیدوارم کیفیت همینجور بمونه
[2]	 label:1 	 text:من وقتی این ایرباد به لپ تاپ ایسوس  وصل میکنم و با نرم افزار حرفه ای   آهنگها را پخش میکنم   صدای گر
[3]	 label:1 	 text:با عرض سلام ضمن عرض خسته نباشید چلوکباب کوبیده سفارشی بسیار عالی بود ولی جوجه در چلو جوجه کباب مخصوص
[4]	 label:1 	 text:خوشمزه و داغ مخصوصا با سس چیلی که فرستاده بودن بسیار خوب بود ممنون ازجوجه داغ و اسنپ
[5]	 label:0 	 text:متاسفانه با وجود تاکید زیاد به خشک و برشته نبودن نان‌ها، نان سنگک به قدری خشک و بیسکویتی بود که قسمت
[6]	 label:0 	 text:طعم چیزکیک اوکی بود عالی نبود ولی اوکی بود اما بسته‌بندی و ارسالش بسیار نامناسب بود کیک کاملا از ریخ
[7]	 label:0 	 text:جوجه کباب خیلی پخته و خشک شده بود 
[8]	 label:0 	 text:سالاد فصل بسیار کیفیت بدی داشت وکاهوداخل ساندویچ از کیفیت خوبی برخوردارنبود
[9]	 label:0 	 text:من  تا مایع ظرفشویی سفارش دادم ولی یکی ارسال شد


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [ ]:
vectorizer = TfidfVectorizer(strip_accents='unicode', min_df=10)
X = vectorizer.fit_transform(persian_dataset['train']['text'])

In [ ]:
vocab=vectorizer.get_feature_names_out()

In [ ]:
print(X.shape)

(55852, 4591)


In [ ]:
doc_vetors=X.toarray()

In [ ]:
# print(docvectors[0])

print([(i,vocab[i],j) for i,j in enumerate(doc_vetors[0]) if j>0])


[(70, 'ادم', 0.18621131199485072), (103, 'از', 0.07056396577345739), (291, 'انتظار', 0.15758240074456317), (337, 'انگار', 0.15329214228657592), (348, 'اهنگ', 0.5095942646638163), (438, 'این', 0.2697752139295794), (475, 'با', 0.16396999851262734), (667, 'برای', 0.11100121633259262), (784, 'بعد', 0.13186176476702166), (845, 'به', 0.13809467213661994), (954, 'تا', 0.1134868216654906), (969, 'تازه', 0.11183745889514918), (1264, 'حالا', 0.17539631713031353), (1274, 'حتما', 0.16466460788199724), (1507, 'دارن', 0.19965633676158456), (1595, 'دستم', 0.13324218159867873), (1797, 'رسیده', 0.20612646352069003), (2553, 'فکر', 0.15832197111586907), (3084, 'موسیقی', 0.25479713233190815), (3266, 'میکنم', 0.14836067780281506), (3269, 'میکنه', 0.17015991334709685), (3679, 'هدست', 0.2735757000931026), (4075, 'پیشنهاد', 0.15626618676707035), (4303, 'کسایی', 0.2377964434781449), (4379, 'که', 0.07239395000134217), (4423, 'کیفیت', 0.07949044900097112), (4574, 'یک', 0.11042471165957501)]


# KMeans Clustering

In [ ]:
from sklearn.cluster import KMeans
from time import time

In [ ]:
item_count=200
km = KMeans(n_clusters=2, init='k-means++', max_iter=100)
t0 = time()
km.fit(X[:item_count])
labels=persian_dataset['train']['label'][:item_count]
print("done in %0.3fs" % (time() - t0))

done in 0.189s


/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [ ]:
from sklearn.metrics import  calinski_harabasz_score

In [ ]:
# Calculate clustering metric
#https://en.wikipedia.org/wiki/Calinski%E2%80%93Harabasz_index

ch_index = calinski_harabasz_score(X[:item_count].toarray(), km.labels_)

print(f"Calinski-Harabasz Index: {ch_index:.2f}")


Calinski-Harabasz Index: 2.48


In [ ]:
import pandas as pd
df = pd.DataFrame({"text":persian_dataset['train']['text'],"label":persian_dataset['train']['label']})

In [ ]:
df_cl=df[:item_count]

In [ ]:
df_cl['cluster'] = km.labels_

df_cl.head()



<ipython-input-28-2b3e031d655a>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cl['cluster'] = km.labels_


,text,label,cluster
0,این هندس فری تازه به دستم رسیده و برای کسایی ...,1,0
1,عالی بود عالی، امیدوارم کیفیت همینجور بمونه,1,1
2,من وقتی این ایرباد به لپ تاپ ایسوس وصل میکنم ...,1,0
3,با عرض سلام ضمن عرض خسته نباشید چلوکباب کوبیده...,1,1
4,خوشمزه و داغ مخصوصا با سس چیلی که فرستاده بودن...,1,1


In [ ]:
# output the result to a text file.

clusters = df_cl.groupby('cluster')

In [ ]:
for cluster in clusters.groups:
    f = open('cluster'+str(cluster)+ '.csv', 'w') # create csv file
    data = clusters.get_group(cluster)[['text']] # get title and overview columns
    f.write(data.to_csv(index_label='id')) # set index to id
    f.close()


In [ ]:
print("Cluster centroids: \n")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(2):
    print("Cluster %d:" % i)
    for j in order_centroids[i, :10]: #print out 10 feature terms of each cluster
        print (' %s' % terms[j])
    print('------------')

Cluster centroids: 

Cluster 0:
 بسیار
 سفارش
 از
 به
 هم
 سرد
 که
 رو
 داده
 با
------------
Cluster 1:
 بود
 خیلی
 عالی
 خوب
 به
 شده
 کیفیت
 خوشمزه
 زمینی
 کاملا
------------


# persian_news_dataset

In [ ]:
dataset = load_dataset("saied/persian_news_dataset", split="train",streaming=True)

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1461: FutureWarning: The repository for saied/persian_news_dataset contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/saied/persian_news_dataset
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [ ]:
dataset

IterableDataset({
    features: ['text', 'title', 'category'],
    n_shards: 1
})

In [ ]:
docs=[]
document_count=1000
counter=0
skip_counter=0
for doc in dataset:
    # print(f"{doc['category']}\t\t{doc['title']}\t\t\t\t{doc['text']}")
    if len(doc['title'])==0 or len(doc['category'])==0:
      skip_counter += 1
      if skip_counter%10000 ==0:
        print(f'{skip_counter} skipped')
      continue
    docs.append(doc)
    counter += 1
    if counter == document_count:
      break
print(len(docs))

10000 skipped
20000 skipped
30000 skipped
40000 skipped
50000 skipped
60000 skipped
70000 skipped
80000 skipped
90000 skipped
100000 skipped
110000 skipped
120000 skipped
130000 skipped
140000 skipped
150000 skipped
160000 skipped
170000 skipped
180000 skipped
190000 skipped
200000 skipped
210000 skipped
1000


In [ ]:
docs[0]

{'text': 'الجزیره: نخست  وزیر رژیم صهیونیستی جنبش حماس را تهدید کرد\n........................................................تهران، خبرگزاری جمهوری اسلامی            06/03/86\nخارجی.سیاسی.غره.اسرائیل\n    بخش انگلیسی زبان شبکه تلویزیونی الجزیره قطر روز یکشنبه گزارش داد:\n"ایهوداولمرت" نخست وزیر رژیم صهیونیستی پس از اصابت موشک های  "قسام"\nمتعلق به گروه مسلح "حماس" اعلام کرد که هیچ یک از اعضای حماس در امان\nنخواهند بود.   الجزیره افزود: پس از آنکه رزمندگان فلسطینی روز یکشنبه دو فروند راکت\nاز غزه به سمت شهر "سدیروت" در جنوب اسراییل پرتاب کردند، اولمرت تهدید\nکرد که دامنه حملات نیروهای اسرائیلی را به غزه گسترش دهد.   وی همچنین در نشست کابینه رژیم صهیونیستی تاکید کرد که هیچ یک از\nاعضای وابسته به حماس درامان نخواهند بود.   به گزارش الجزیره: گردانهای عزالدین قسام شاخه نظامی حماس مسوولیت\nحمله موشکی امروز به جنوب فلسطین اشغالی را که به کشته شدن یک صهیونیست\nمنجر شد بر عهده گرفت.   براساس بیانیه گردانهای عزالدین قسام که برای رسانه ها ارسال شده است\nرزمندگان این گروه ساعت 15/9 دقیقه صبح امروز

In [ ]:
len(set([doc['category'] for doc in docs]))

6

# Preprocess

In [ ]:
!pip install stanza

In [ ]:
import stanza
stanza.download('fa')       # This downloads the English models for the neural pipeline

INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: fa (Persian) ...
INFO:stanza:File exists: /root/stanza_resources/fa/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources


In [ ]:
nlp = stanza.Pipeline(lang='fa', processors='tokenize,pos')

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: fa (Persian):
| Processor | Package      |
----------------------------
| tokenize  | perdt        |
| mwt       | perdt        |
| pos       | perdt_charlm |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!


In [ ]:
doc = nlp("محسن محمدي متولد شهر تهران است. او در ۲۰ سالگی در مشهد به دانشگاه رفت.")

In [ ]:
doc

[
  [
    {
      "id": 1,
      "text": "محسن",
      "upos": "PROPN",
      "xpos": "N_ANM",
      "feats": "Number=Sing",
      "start_char": 0,
      "end_char": 4
    },
    {
      "id": 2,
      "text": "محمدي",
      "upos": "PROPN",
      "xpos": "N_ANM",
      "feats": "Number=Sing",
      "start_char": 5,
      "end_char": 10
    },
    {
      "id": 3,
      "text": "متولد",
      "upos": "ADJ",
      "xpos": "ADJ_AJP",
      "start_char": 11,
      "end_char": 16
    },
    {
      "id": 4,
      "text": "شهر",
      "upos": "PROPN",
      "xpos": "N_IANM",
      "feats": "Number=Sing",
      "start_char": 17,
      "end_char": 20
    },
    {
      "id": 5,
      "text": "تهران",
      "upos": "PROPN",
      "xpos": "N_IANM",
      "feats": "Number=Sing",
      "start_char": 21,
      "end_char": 26
    },
    {
      "id": 6,
      "text": "است",
      "upos": "AUX",
      "xpos": "AUX",
      "feats": "Number=Sing|Person=3",
      "start_char": 27,
      "end_char": 30,

In [ ]:
print(*[f'word: {word.text+" "}\tpos: {word.pos}' for sent in doc.sentences for word in sent.words], sep='\n')

word: محسن 	pos: PROPN
word: محمدي 	pos: PROPN
word: متولد 	pos: ADJ
word: شهر 	pos: PROPN
word: تهران 	pos: PROPN
word: است 	pos: AUX
word: . 	pos: PUNCT
word: او 	pos: PRON
word: در 	pos: ADP
word: ۲۰ 	pos: NUM
word: سالگی 	pos: NOUN
word: در 	pos: ADP
word: مشهد 	pos: PROPN
word: به 	pos: ADP
word: دانشگاه 	pos: NOUN
word: رفت 	pos: VERB
word: . 	pos: PUNCT


Tokenize and keep NOUN,VERB,ADV AND ADV

In [ ]:
dataset = load_dataset("saied/persian_news_dataset", split="train", streaming=True)
first_doc = next(iter(dataset))
doc = nlp(first_doc['text'])

pos_to_keep = {"NOUN", "VERB", "ADV", "ADJ"}

filtered_words = [(word.text, word.pos) for sent in doc.sentences for word in sent.words if word.pos in pos_to_keep]

for word, pos in filtered_words:
    print(f'Word: {word}\tPOS: {pos}')

Word: سخنگوی	POS: NOUN
Word: گفت	POS: VERB
Word: روند	POS: NOUN
Word: رعایت	POS: NOUN
Word: کردن	POS: NOUN
Word: گونه‌ای	POS: NOUN
Word: امروز	POS: NOUN
Word: شاهد	POS: ADJ
Word: هستیم	POS: VERB
Word: مردم	POS: NOUN
Word: همچنان	POS: ADV
Word: پارک‌ها	POS: NOUN
Word: حضور	POS: NOUN
Word: یابند	POS: VERB
Word: وضعیت	POS: NOUN
Word: قرمز	POS: ADJ
Word: می	POS: NOUN
Word: رسیم	POS: VERB
Word: گزارش	POS: NOUN
Word: عصر	POS: NOUN
Word: امروز	POS: NOUN
Word: گفت‌وگوی	POS: NOUN
Word: مجاری	POS: NOUN
Word: خبرنگاران	POS: NOUN
Word: بیان	POS: NOUN
Word: اینکه	POS: NOUN
Word: مردم	POS: NOUN
Word: اخبار	POS: NOUN
Word: مربوط	POS: ADJ
Word: تنها	POS: ADV
Word: سایت‌های	POS: NOUN
Word: رسمی	POS: ADJ
Word: رسانه	POS: NOUN
Word: ملی	POS: ADJ
Word: پیگیری	POS: NOUN
Word: کنند	POS: VERB
Word: اظهار	POS: NOUN
Word: کرد	POS: VERB
Word: هرگونه	POS: NOUN
Word: خبری	POS: NOUN
Word: فضای	POS: NOUN
Word: مجازی	POS: ADJ
Word: منتشر	POS: ADJ
Word: می‌شود	POS: VERB
Word: صحت‌وسقم	POS: NOUN
Word: آگاه	POS: ADJ
Wo

Tokenize

In [ ]:
dataset = load_dataset("saied/persian_news_dataset", split="train", streaming=True)
first_doc = next(iter(dataset))
doc = nlp(first_doc['text'])

print(*[f'word: {word.text+" "}\tpos: {word.pos}' for sent in doc.sentences for word in sent.words], sep='\n')

word: اهواز 	pos: PROPN
word: - 	pos: PUNCT
word: سخنگوی 	pos: NOUN
word: ستاد 	pos: PROPN
word: مدیریت 	pos: PROPN
word: بیماری 	pos: PROPN
word: ناشی 	pos: PROPN
word: از 	pos: ADP
word: ویروس 	pos: PROPN
word: کرونای 	pos: PROPN
word: خوزستان 	pos: PROPN
word: گفت 	pos: VERB
word: : 	pos: PUNCT
word: اگر 	pos: SCONJ
word: روند 	pos: NOUN
word: رعایت 	pos: NOUN
word: کردن 	pos: NOUN
word: به 	pos: ADP
word: گونه‌ای 	pos: NOUN
word: باشد 	pos: AUX
word: که 	pos: SCONJ
word: امروز 	pos: NOUN
word: شاهد 	pos: ADJ
word: آن 	pos: PRON
word: هستیم 	pos: VERB
word: و 	pos: CCONJ
word: مردم 	pos: NOUN
word: همچنان 	pos: ADV
word: در 	pos: ADP
word: پارک‌ها 	pos: NOUN
word: حضور 	pos: NOUN
word: یابند 	pos: VERB
word: و 	pos: CCONJ
word: . 	pos: PUNCT
word: . 	pos: PUNCT
word: . 	pos: PUNCT
word: به 	pos: ADP
word: وضعیت 	pos: NOUN
word: قرمز 	pos: ADJ
word: می 	pos: NOUN
word: رسیم 	pos: VERB
word: . 	pos: PUNCT
word: به 	pos: ADP
word: گزارش 	pos: NOUN
word: ، 	pos: PUNCT
word: رضا 	pos: PR

Third Part of the exercise

In [ ]:
!pip install datasets
!pip install stanza

In [ ]:
import stanza
from datasets import load_dataset

In [ ]:
stanza.download('fa')
nlp = stanza.Pipeline(lang='fa', processors='tokenize')
dataset = load_dataset("saied/persian_news_dataset", split="train", streaming=True)

INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: fa (Persian) ...
INFO:stanza:File exists: /root/stanza_resources/fa/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: fa (Persian):
| Processor | Package |
-----------------------
| tokenize  | perdt   |
| mwt       | perdt   |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Done loading processors!
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/datasets/load.py:1461: FutureWarning: The repository for saied/persian_news_dataset contain

In [ ]:
tokenized_docs = []
for doc in dataset:
    tokenized_doc = nlp(doc['text'])
    tokenized_text = ' '.join([word.text for sent in tokenized_doc.sentences for word in sent.words])
    tokenized_docs.append(tokenized_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(strip_accents='unicode', min_df=10)
X = vectorizer.fit_transform(tokenized_docs)

In [ ]:
from sklearn.cluster import KMeans
from time import time

km = KMeans(n_clusters=2, init='k-means++', max_iter=100)
t0 = time()
km.fit(X)
print("Clustering done in %0.3fs" % (time() - t0))

In [ ]:
from sklearn.metrics import calinski_harabasz_score

ch_index = calinski_harabasz_score(X.toarray(), km.labels_)
print(f"Calinski-Harabasz Index: {ch_index:.2f}")


In [ ]:
import pandas as pd

df_cl = pd.DataFrame({"text": [doc['text'] for doc in dataset], "label": [doc['label'] for doc in dataset]})
df_cl['cluster'] = km.labels_


In [ ]:
# Output the result to a text file for each cluster
clusters = df_cl.groupby('cluster')
for cluster in clusters.groups:
    with open('cluster' + str(cluster) + '.csv', 'w') as f:
        data = clusters.get_group(cluster)[['text']]
        f.write(data.to_csv(index_label='id'))


In [ ]:
# Output cluster centroids
print("Cluster centroids: \n")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(2):
    print("Cluster %d:" % i)
    for j in order_centroids[i, :10]:  # Print out 10 feature terms of each cluster
        print(' %s' % terms[j])
    print('------------')